In [1]:
import mailbox
from bs4 import BeautifulSoup
import re

documents = []
mbox = mailbox.mbox(r"C:\Users\Nathan\Downloads\takeout-20251111T021856Z-1-001\Takeout\Mail\homework.mbox")

for message in mbox:
    print(message["subject"])

    payload = message.get_payload()

    # === MULTIPART EMAIL (HTML + attachments + text) ===
    if isinstance(payload, list):
        for part in message.walk():
            content_type = part.get_content_type()
            content_disposition = str(part.get("Content-Disposition", ""))

            # Only handle readable parts
            if content_type in ["text/plain", "text/html"] and "attachment" not in content_disposition:
                raw_bytes = part.get_payload(decode=True)
                if raw_bytes:
                    try:
                        text = raw_bytes.decode(part.get_content_charset() or "utf-8", errors="ignore")
                    except Exception:
                        text = str(raw_bytes)

                    soup = BeautifulSoup(text, "lxml")
                    text = soup.get_text(separator="\n", strip=True)
                    cleaned_text = re.sub(r'=(0D|0A|09|E2|80|8C)', '', text)
                    cleaned_text = cleaned_text.replace("=", "")
                    documents.append(cleaned_text)

    # === SIMPLE EMAIL (no MIME parts) ===
    elif isinstance(payload, str):
        soup = BeautifulSoup(payload, "lxml")
        text = soup.get_text(separator="\n", strip=True)
        cleaned_text = re.sub(r'=(0D|0A|09|E2|80|8C)', '', text)
        cleaned_text = cleaned_text.replace("=", "")
        documents.append(cleaned_text)

    else:
        raise TypeError("Unexpected payload type")

    print(" ")

print(f"\n✅ Extracted {len(documents)} messages.")


Halloween Information
 
IMPORTANT (PLEASE READ) Safety & Cleaning Inspections
 
Chapter Seventeen: In Which The Corn Surrenders To Us
 
Chapter Eighteen: In Which Teaching Is Done
 
Nathan, =?UTF-8?B?eW914oCZcmUgaW4hIPCfjok=?= famfam sibling secret
 santa starts now!
 
=?iso-8859-1?Q?Save_on_Apple=AE!_USU_Campus_Store_Fall_Tech_Sale?=
 
Papa Murphy's Order Received
 
Parking at Aggie Village 
 
Final Updated List for Trick-or-Treat
 
Thank you for your Steam purchase!
 
Nathan, your order is confirmed.
 
=?UTF-8?Q?Join_us_for_Homecoming_=E2=80=93_Discounted_Tickets_for_Alumni!?=
 

✅ Extracted 23 messages.


In [51]:
import os
import numpy as np
from voyageai.client import Client
from google import genai


vo = Client(api_key="REDACTED")
google_api_key = "REDACTED"
client = genai.Client(api_key=google_api_key)

In [62]:
def answer_query(query: str):
    # 1️⃣ Embed the query
    query_embedding = vo.embed([query], model="voyage-3.5", input_type="query").embeddings[0]

    # 2️⃣ Compute cosine similarity
    def cosine_sim(a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

    similarities = [cosine_sim(query_embedding, doc_emb) for doc_emb in documents_embeddings]

    # 3️⃣ Find best matching document
    best_idx = int(np.argmax(similarities))
    best_doc = documents[best_idx]

    # 4️⃣ Send query + document to Google Gemini
    user_query = f"""
    Here is question from the user: {query}.
    
    Here is the closest matching document related to the question:
    <document>
    {best_doc}
    </document>

    Use <document> to help answer the question.
    """ 

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        config=genai.types.GenerateContentConfig(
            system_instruction="You are a helpful email assistant answering questions about emails."
        ),
        contents=user_query
    )

    # Gemini returns content as a list of dicts
    return response.text


In [63]:

# Example usage
print(answer_query("What did Alyssa do on Halloween?"))

Based on the provided document, there is no information about what Alyssa did on Halloween. The document discusses a Halloween activity in the Family Housing Community where units handed out candy to trick-or-treaters.


In [64]:
### This makes sense because I actually have two different Halloween related emails.

In [65]:
print(answer_query("How many people are on the trick-or-treat list?"))

There are 20 confirmed trick-or-treaters on the list, with the possibility that more may join later that evening.


In [66]:
print(answer_query("What did I order from Dominos?"))

Based on the document provided, you did not order from Domino's. The document is an order confirmation from **Subway® Restaurants**.

From Subway, you ordered:
*   **Footlong Regular Sub Spicy Italian** (Italian Herbs & Cheese, Toasted, Monterey Cheddar, Lettuce, Spinach, Pickles, Black Olives, Regular Mayonnaise, Yellow Mustard, Black Pepper)
*   **Miss Vickie’s® Jalapeño** (chips)
*   **Cookie** (Chocolate Chip)


In [67]:
print(answer_query("Sorry, I meant what did I order from Papa Murphys?"))

Based on the order confirmation, you ordered:

*   **1 x MySLICE Reward $1 Medium 1-Topping Original Medium (12") Pizza** with **Pepperoni - Regular**.


In [68]:
print(answer_query("What is Alyssa's new assignment in Spanish Fork?"))

The provided document does not mention an "Alyssa" or her new assignment.

However, the person who wrote the email, **Sister Wessman**, started a new assignment on Monday at the **ELE preschool in Spanish Fork**. It's a small preschool for children from vulnerable homes.


In [69]:
print(answer_query("When are the cleaning and safety inspections for Aggie Village, and who will be doing them?"))

Cleaning and safety inspections for Aggie Village will be conducted on **Wednesday, November 5, starting at 2:00 PM**.

The inspections will be carried out by **Judson and Jacey (the RAs), along with Richard (the Area Coordinator)**.


In [70]:
print(answer_query("What did I buy from the Steam store again?"))

Based on the document, you bought **The Jackbox Party Pack 7**.

It was purchased on October 15, 2025, for a subtotal of $17.99 (totaling $19.30 with tax).
